# CODICE PULITO di visualizzazione_dati_test
1. modifiche file singoli file merge (es. conversioni dummy, round age)
2. merge outer
3. creazione colonna globale per variabile comune ai due dataset
4. studio e applicazione strategia dove ci sono le differenze
5. ricordati ritrasformare dummy e cancellare x e y

appuntino --> age precedenza PTDEMOG


In [1]:
import pandas as pd

In [2]:
# --- 1. Caricamento dei dataset puliti ---
adnimerge = pd.read_csv("ADNIMERGE_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp"])
ptdemog = pd.read_csv("PTDEMOG_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp", "PTDOB"])

## Riconversione dummys

In [22]:
cols_x = sorted([c for c in ptdemog.columns if c.endswith('_0')])
coppie = []
for c in cols_x:
    base = c[:-2]
    c_y = base + '_1'
    if c_y in ptdemog.columns:
        coppie.append(base)

print("Coppie trovate:", coppie)

Coppie trovate: ['ETHNICITY', 'MARRY', 'RACE']


In [ ]:
# --- ADNIMERGE ---
for prefix, categories in [
    ("MARRY", [0, 1, 2, 3]),
    ("ETHNICITY", [0, 1]),
    ("RACE", [0, 1, 2, 3, 4, 5]),
]:
    cols_dummy = [f"{prefix}_{cat}" for cat in categories]
    ricostruita = pd.from_dummies(adnimerge[cols_dummy], sep="_", default_category="missing")
    adnimerge[prefix] = ricostruita[prefix]
    adnimerge = adnimerge.drop(columns=cols_dummy)

In [26]:
# --- PTDEMOG ---
for prefix, categories in [
    ("MARRY", [0, 1, 2, 3]),
    ("ETHNICITY", [0, 1]),
    ("RACE", [0, 1, 2, 3, 4, 5]),
]:
    cols_dummy = [f"{prefix}_{cat}" for cat in categories]
    ricostruita = pd.from_dummies(ptdemog[cols_dummy], sep="_", default_category="missing")
    ptdemog[prefix] = ricostruita[prefix]
    ptdemog = ptdemog.drop(columns=cols_dummy)

In [85]:
for prefix in ["RACE", "ETHNICITY", "MARRY", "GENDER"]:
    if prefix in adnimerge.columns:
        adnimerge[prefix] = adnimerge[prefix].replace("missing", pd.NA)
    if prefix in ptdemog.columns:
        ptdemog[prefix] = ptdemog[prefix].replace("missing", pd.NA)

## Merge

In [86]:
keys = ["RID", "EXAMDATE"]

# Conto delle combinazioni uniche di chiavi
left_keys = adnimerge[keys].drop_duplicates()
right_keys = ptdemog[keys].drop_duplicates()

key_match = left_keys.merge(
    right_keys,
    on=keys,
    how="outer",
    indicator=True
)

counts = key_match["_merge"].value_counts()
print("Conteggio chiavi uniche per [RID, EXAMDATE]:")
print(counts)

print(f"Match: {counts.get('both', 0)}")
print(f"Solo in adnimerge: {counts.get('left_only', 0)}")
print(f"Solo in ptdemog: {counts.get('right_only', 0)}")

Conteggio chiavi uniche per [RID, EXAMDATE]:
_merge
left_only     8861
right_only    5515
both           445
Name: count, dtype: int64
Match: 445
Solo in adnimerge: 8861
Solo in ptdemog: 5515


In [88]:
# --- 2. Merge su RID + EXAMDATE ---
merged = pd.merge(
    adnimerge,
    ptdemog,
    on=keys,
    how="outer",
    indicator=True
)

In [89]:
# --- 3. Log di controllo post-merge ---
print(merged["_merge"].value_counts())
#merged = merged.drop(columns="_merge") #tenere per le visualizazioni eliminare solo alla fine

_merge
left_only     8861
right_only    5515
both           445
Name: count, dtype: int64


In [90]:
merged = merged[sorted(merged.columns)]
merged

,ADAS11,ADAS13,AGE_bl,AGE_x,AGE_y,APOE4,CDRSB,COLPROT,DX,DX_0,...,RAVLT_immediate,RID,VISCODE_x,VISCODE_y,VISIT_MONTH_x,VISIT_MONTH_y,Ventricles,_merge,update_stamp_x,update_stamp_y
0,NaN,NaN,NaN,NaN,60.711841,NaN,NaN,NaN,NaN,NaN,...,NaN,1,NaN,f,NaN,0.0,NaN,right_only,NaT,2005-08-18 00:00:00
1,NaN,NaN,NaN,NaN,74.379192,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,sc,NaN,0.0,NaN,right_only,NaT,2005-08-17 00:00:00
2,10.67,18.67,74.3,74.3,NaN,0.0,0.0,ADNI1,NaN,1.0,...,44.0,2,bl,NaN,0.0,NaN,118233.0,left_only,2023-07-07 04:59:40,NaT
3,NaN,NaN,NaN,NaN,79.477070,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,sc,NaN,61.0,NaN,right_only,NaT,2013-03-22 15:23:58
4,NaN,NaN,NaN,NaN,80.468172,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,m72,NaN,73.0,NaN,right_only,NaT,2013-05-30 10:05:05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14816,NaN,NaN,NaN,NaN,74.340862,NaN,NaN,NaN,NaN,NaN,...,NaN,10891,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14817,NaN,NaN,NaN,NaN,76.585900,NaN,NaN,NaN,NaN,NaN,...,NaN,10893,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14818,NaN,NaN,NaN,NaN,79.600274,NaN,NaN,NaN,NaN,NaN,...,NaN,10894,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14819,NaN,NaN,NaN,NaN,67.356605,NaN,NaN,NaN,NaN,NaN,...,NaN,10895,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48


## Creazione colonna unificata

In [91]:
both_rows = merged[sorted(merged.columns)]
both_rows[both_rows["_merge"] == "both"]

,ADAS11,ADAS13,AGE_bl,AGE_x,AGE_y,APOE4,CDRSB,COLPROT,DX,DX_0,...,RAVLT_immediate,RID,VISCODE_x,VISCODE_y,VISIT_MONTH_x,VISIT_MONTH_y,Ventricles,_merge,update_stamp_x,update_stamp_y
77,2.0,4.0,72.6,77.552772,77.678303,0.0,0.0,ADNIGO,NaN,1.0,...,58.0,21,m60,sc,59.0,60.0,18783.0,both,2023-07-07 04:59:41,2014-07-10 19:03:08
78,3.0,5.0,72.6,78.560301,78.685832,0.0,0.0,ADNI2,NaN,1.0,...,53.0,21,m72,m72,72.0,72.0,22013.0,both,2023-07-07 04:59:41,2013-05-30 10:05:05
96,5.0,10.0,71.7,76.817043,76.969199,0.0,0.0,ADNIGO,NaN,1.0,...,42.0,23,m60,sc,61.0,62.0,28003.0,both,2023-07-07 04:59:41,2013-03-22 15:23:58
97,6.0,10.0,71.7,77.813621,77.965777,0.0,0.0,ADNI2,NaN,1.0,...,39.0,23,m72,m72,73.0,74.0,29028.0,both,2023-07-07 04:59:41,2013-05-30 10:05:05
124,5.0,10.0,77.7,82.803354,82.915811,0.0,0.0,ADNIGO,NaN,1.0,...,53.0,31,m60,sc,61.0,62.0,31653.0,both,2023-07-07 04:59:41,2014-01-15 19:03:02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6833,8.0,12.0,70.2,71.182888,71.321013,0.0,1.0,ADNI2,NaN,0.0,...,32.0,2396,m12,m12,12.0,13.0,49461.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05
6842,14.0,26.0,71.5,72.529432,72.646133,2.0,4.0,ADNI2,NaN,0.0,...,28.0,2398,m12,m12,12.0,13.0,59320.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05
6851,14.0,24.0,79.1,80.118480,80.298426,0.0,4.5,ADNI2,NaN,0.0,...,26.0,2403,m12,m12,12.0,14.0,39370.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05
6859,10.0,16.0,71.6,72.596578,72.687201,0.0,0.5,ADNI2,NaN,0.0,...,47.0,2405,m12,m12,12.0,13.0,24946.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05


In [105]:
for base in coppie:
    col_x, col_y = f"{base}_x", f"{base}_y"

    if base == "update_stamp":
        continue  # non toccare update_stamp_x/_y, restano separate

    nuova_colonna = f"{base}_merged"

    if base == "AGE":
        merged[nuova_colonna] = merged[col_y].combine_first(merged[col_x])   # priorità a PTDEMOG
    else:
        merged[nuova_colonna] = merged[col_x].combine_first(merged[col_y])   # priorità ad ADNIMERGE

KeyError: 'ETHNICITY_x'

In [101]:
cols_da_rimuovere = []
for base in coppie:
    if base == "update_stamp":
        continue  # la lasci intatta, come richiesto
    cols_da_rimuovere += [f"{base}_x", f"{base}_y"]

merged = merged.drop(columns=cols_da_rimuovere)

In [103]:
display_cols = ['RID', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'ETHNICITY','RACE', 'AGE', '_merge']
merged[display_cols].head(20)

KeyError: "['VISCODE', 'VISIT_MONTH', 'AGE'] not in index"

In [104]:
merged

,ADAS11,ADAS13,AGE_bl,AGE_x,AGE_y,APOE4,CDRSB,COLPROT,DX,DX_0,...,VISCODE_y,VISIT_MONTH_x,VISIT_MONTH_y,Ventricles,_merge,update_stamp_x,update_stamp_y,ETHNICITY,MARRY,RACE
0,NaN,NaN,NaN,NaN,60.711841,NaN,NaN,NaN,NaN,NaN,...,f,NaN,0.0,NaN,right_only,NaT,2005-08-18 00:00:00,NaN,1,NaN
1,NaN,NaN,NaN,NaN,74.379192,NaN,NaN,NaN,NaN,NaN,...,sc,NaN,0.0,NaN,right_only,NaT,2005-08-17 00:00:00,0,1,5
2,10.67,18.67,74.3,74.3,NaN,0.0,0.0,ADNI1,NaN,1.0,...,NaN,0.0,NaN,118233.0,left_only,2023-07-07 04:59:40,NaT,0,1,5
3,NaN,NaN,NaN,NaN,79.477070,NaN,NaN,NaN,NaN,NaN,...,sc,NaN,61.0,NaN,right_only,NaT,2013-03-22 15:23:58,0,3,5
4,NaN,NaN,NaN,NaN,80.468172,NaN,NaN,NaN,NaN,NaN,...,m72,NaN,73.0,NaN,right_only,NaT,2013-05-30 10:05:05,0,3,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14816,NaN,NaN,NaN,NaN,74.340862,NaN,NaN,NaN,NaN,NaN,...,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48,0,1,5
14817,NaN,NaN,NaN,NaN,76.585900,NaN,NaN,NaN,NaN,NaN,...,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48,0,3,5
14818,NaN,NaN,NaN,NaN,79.600274,NaN,NaN,NaN,NaN,NaN,...,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48,0,3,5
14819,NaN,NaN,NaN,NaN,67.356605,NaN,NaN,NaN,NaN,NaN,...,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48,0,1,5


In [83]:
missing_report = merged.isna().sum().sort_values(ascending=False)
print(missing_report)

PTADBEG            14445
DX                 14302
HAS_QC_ERROR       13543
PTCOGBEG           12295
EDUCATION_y         8882
VISCODE_y           8863
VISIT_MONTH_y       8862
AGE_y               8862
MARRY_y             8861
PTDOB               8861
PTID                8861
RACE_y              8861
ETHNICITY_y         8861
GENDER_y            8861
update_stamp_y      8861
MidTemp             7075
Entorhinal          7075
Fusiform            7075
Hippocampus         6684
Ventricles          6094
APOE4               5707
ADAS13              5641
FAQ                 5621
RAVLT_immediate     5611
CDRSB               5601
ADAS11              5565
MMSE                5544
AGE_bl              5521
AGE_x               5521
VISIT_MONTH_x       5515
EDUCATION_x         5515
ETHNICITY_x         5515
COLPROT             5515
GENDER_x            5515
FSVERSION           5515
DX_0                5515
DX_2                5515
DX_1                5515
MARRY_x             5515
ICV                 5515


In [84]:
missing_count = merged.isna().sum()
missing_pct = (missing_count / len(merged) * 100).round(2)

missing_report = pd.DataFrame({
    "n_missing": missing_count,
    "pct_missing": missing_pct
}).sort_values("n_missing", ascending=False)

print(missing_report)

                 n_missing  pct_missing
PTADBEG              14445        97.46
DX                   14302        96.50
HAS_QC_ERROR         13543        91.38
PTCOGBEG             12295        82.96
EDUCATION_y           8882        59.93
VISCODE_y             8863        59.80
VISIT_MONTH_y         8862        59.79
AGE_y                 8862        59.79
MARRY_y               8861        59.79
PTDOB                 8861        59.79
PTID                  8861        59.79
RACE_y                8861        59.79
ETHNICITY_y           8861        59.79
GENDER_y              8861        59.79
update_stamp_y        8861        59.79
MidTemp               7075        47.74
Entorhinal            7075        47.74
Fusiform              7075        47.74
Hippocampus           6684        45.10
Ventricles            6094        41.12
APOE4                 5707        38.51
ADAS13                5641        38.06
FAQ                   5621        37.93
RAVLT_immediate       5611        37.86
